# Decompose the layer-1 update

Attention versus MLP contributions to the post-layer-1 trajectory.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
names=["blocks.1.hook_resid_pre","blocks.1.hook_resid_mid","blocks.1.hook_resid_post"]
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=names)
pre,mid,post=[c[n][0].detach().cpu().numpy() for n in names]
print("attention MSE",float(np.mean((mid-pre)**2)))
print("MLP MSE",float(np.mean((post-mid)**2)))
print("total MSE",float(np.mean((post-pre)**2)))
plot_trajectory(shared_pca([pre,mid,post]),["pre layer 1","post attention","post layer 1"],"Layer-1 update decomposition","figures/layer1_decomposition.png")

attention MSE 3.6330502033233643
MLP MSE 5.910215377807617
total MSE 14.173662185668945
